# Genesis Studio Export on a free GPU
Runtime menu -> Change runtime type -> **T4 GPU**, then run each cell top to bottom (about 10 min the first time). Models download from GitHub straight onto Google's machine, nothing is stored on your phone.

Licences: BiRefNet = MIT (fine for a paid product). MatAnyone 2 = non-commercial, reference/benchmark only.

In [ ]:
!nvidia-smi -L

In [ ]:
!pip -q install timm kornia einops scipy scikit-image imageio imageio-ffmpeg opencv-python-headless hydra-core omegaconf cython gitpython huggingface_hub safetensors tqdm

In [ ]:
import base64, io, zipfile, os
B64 = '''UEsDBBQAAAAIAIc5al3mW0/HrhIAAAcyAAAQAAAAc3R1ZGlvX2V4cG9ydC5wea1a/XPbuNH+3X8FqptOyRxFS06cS5XTvZNLfInf8dkeO9e+HY+HhUhQ4pkieSRlyc3kf3+fXQD8kCU77dQzligQWCwWu89+AN/96XBVlYezJDtU2b0oHupFnr08GAwG1/UqSnJxsinysp6IRTJfqKoe/rGSaVI/iEKVVZ6JpazrJJuLOC+FFFcn7y+uPpx8EGGaFMIpV1kl0EmKj5e/iVm+8USW1yLJRL1QYlbm60qVrn9w8HmdC5XNk0xVwrkf+W+O3MmBEMOhbgQ5JTPhRCqWq7T2hLpXJViQZS1+Pf18+K6Q4UId/nz9wRMx9dfcFDKJRFHm0SqsXZCjv5+TKxWfq1o4GOgKmRYLSSye/O3k6h8iLuVSieFPIi/qJJTpME7ztajVEjKQqYhXVYK+jm3wiweXelehwqRJVqt5KWvuwk2BbVLUs7sgiE1mD3mmjoRzdfLLydXJ+fsTcXF+9o+J+FXW7/idOBJJJVZZpEoW2Pnn38T18EzOIMVsGObLpSrDBHylCWYL1aM1xklZ1ZiruiMuu3RJQFqIvGRf/FZhAbWoc7FUslqVSiyw8jDN0U5z6x0w3M9VXb0VUc67WS2w1UmNjV7Vxao2PCSZ3QBs8T34wzpSVVXiIV+BdBrh9bpM6lpl4vFKwFW+5HnlCgpZQisqpTBjWB3+evHh5Cz4cPL+9Pr04jy4HwVvjvxlBDU6y/O7agIhp3gwm2J1xhVTWjR2I4dAMWWaryCCqk6Wese+t22hZJnV4QJtKbS+BqOyoA70plrICJL5XmBfk0x3fGsWbacuUno15QXQUqsafYmQVrdZqrIIWv9bJeeK9FxbHbqRxQWKLQ4aI4YJG5K/VrOlGOYkYX9ZvMI0MxnemaXwOv3fi/m/QYj5MKRAdLjMI6Ub/2vcoJkUj0mn0JhMkQVR42ZYJf9SYjx6Mzr4ROKCleQsq1Kxrq/z8k6VEwE4Of9wchW8//XD9C9PcPUlyb4SQ1/A0dc+O4eFrBeHDVN/ObhaZSJW2LOAGav8agHzJ40jW/HFZZnPS9LTlMGIZ4xAV0iYxP9eX5xPxJdBYToNJiP/1ZE3qGps5GAyMGA4+OoffFLy/kEkhBMwDRqcZFUCKcerLCSFq0SVW/QM0nyehLQaUHqoADj4hpaIdYJFr8gsy3DhEyofaJKgOAf8VcoTvwOGPZFXnqhWM3AWgjE8P+ADyq0ODj4BXaCMeeWTMPwoKTOYvGN/y1lF304QxEmqgsB1DzBYvwPPqqydkSeIiGsnz1ZL8CorkRW2qbcSenVm3zToicbPtnELHund9enBwd9PTj9++nwNbr80wpyIgQW0oWnzi3ox8MSASMGy6l4f22g7GdXr9TFttgscmuq9pwZ++fXg4AAIIir54Lx4cbdmtwQTKcG6Q5L3I8iicvAGriddVYvp53IFJNbDSiWjgBG2ckigHuS0CUgPDCEjjvD+iH+GwJkp/fL/hj75e1nUgGIeqsE9iRlz0c9PqosCy4gcQ4r+sG7g9fVDBaGfbJLaGYQyowE5ukIDgc/inigPNLm4qGg+UAOkOzTv+3eXweXVxWXwy+W1K+AlXo78UXdqZyx+5HE/ivHRuDO3ptV016tGy80t/14voF2ChNMOye88MZuXhgUSluM2L818+V3bn/5m6HbXtECiawwHERiyLNTN5Oi2eUmzL5PMGfujVvDikNtooOvSJM2Ln8RIABGATGYJhgteqj/aYkOzDYkBCABnDho84ZBaMPI4a/ECIw/FkYuHI2hH+2rRf6XfqbLIU3ZFU6J6ev4ZyPfu6uRdKxFibhcX+LxBgCaG+P+zOPLEBEIZ4h8/WmnoDfFlAVWIeK/D+/o9fF6peefdvzi7uAp+/nh1dPXxZ9e1Ook1wv1XyukpoSb4lPJ1FE4sCC5yM8ioX6mg3plp80iDjN0wGAfkPxx+ie0jtA4yfo4UBRQuHMR33UDKBCd5lj4wdaBlEwkBF2dylnDkuoSRUfxDkVAT8HF8oyKxwla84ZDJJ7Tt2ChjsNeY6iwpO6D6ew6dIowEmtxDvnlJuGKnN8vdAaugYiyRAh7jktCo4gxcm5mbVWC9Wf6HbAes6iStGhBZqPAugOeoVRAlYb87kZs2lJzZLCggfQphVDT9RUKzDI8RxS3sbtIc9rhrgWtFwUyFFRq8vmmRmC2q2SsxnYoGyLVxtS9vXbLKAi4j1Ho/CIuVERXY5fk7y3G21+dUket2Fkdj6twx2uGre5kahTUooUPdPk5s+lZstM0Zj45eeYI+n7NOH0HhQ6GcrPCRLMj6Jawapn18bECEJnA2MEd0kGUJP3KDoOHNsSfwdfyav0avbz3RIUAUut2Pjv5K/Y6IJ/o67nfneepm30gzAnbRzsbHrmRVgSjeAS5A4caue3MOzb/tiErjM0INQyDJYlVSGM4xUte/QGciLWmndm+G41u/SubLPAFua27wjU3Ep57fvcGUIy1pCgMx0CKInpsVJFxFctBOopmgRh+hQ/0QhJTdma0kQ+5vGfGEHVt7YvHEXp2dnp+8u+rBzpnPRh90wIaoW99NamsUovIYEYLVGws+kH+wluVyVSAZRYZJqJRgUnzb4NCIDSjSTbxyJF3in5rqP4WTJkgp8lgAbjX0uBQCljqlZkjQ81IgqgfdjG59ccUroCBVD2fZm+zC+eT93d0JXtywlM/CVsPv0X7gWsoubjUg3OpOR4vCnMJfzcipbX2Pxp0UGNMoHglM6qYjdUuAXrS99bsn6LRc9LASAWBgN4yesYeU2msJ8XTTnTM9g4eDzvwUPrqNs7LKG4JwTwYO04VfiedTfvTxpLsvKX7R+99YDfNMoUprL0vm1a7HWbaqqL8bp82KunusWT8NNurMX4YPKiDsQJdla/RbYFKpPyjyazT2FoGONRjOqqvaWJbuTzlOEymG9aaZDel/Tmm3oR4Q1E6tCUb8y7BFjIxfu7ugRTueJwiy48GSM0q6eu7wCVT0iNFO+IuIok4QxJRU+1CQkKLMxoEo3H7Qlizn5BMeiTMuXb9QEFHdgrUVsHUpjyTdEI0xOy15K07tYCZpn4/grHDAgUY05I2z31VYV9Ob8a37jQM1XvL2URARmYynO1Clmp8fp822fytb30R9Ow7eR26XjH56iieqaZgImQnpmlZAtIM67ziJ3e5upwjZGzgOph52jWBMm5qqzBpCzzdhXuOGqEimAo6hTQqp+wflfMYhcz+TTJZyrpI8iONloeaPoh8eRi6kHwWFS3LrN/3BjMH6MVAb1vjB8IFQbogsP1X3KqUfqiy12xjG9FnKtc4wqaVINkG8rLl9Pjt6xY0ElPHgy/rr5sviK7eUugWLmfgvY7R1xTgYJtyJP2TGX+HkXufts83Ra00VulIpnmmpomS11P1K5mn8epubh9X9q6NRwc3L/D5O5Zzx+/sY2MAeGL9I3Fo+lJW31RX/ktJpB0LzqDqUZNPuu9PLE+MagQgxgUEr+U4k5fNIn7eXgkdZhXlWJ/NVvqp04BeTsc8ealVZxbKDuCzbJmOFv0boDZD4U8/8H+djOhMDhpEfiCUS8mjQhDurTSCptuZUZejppM0jPWzjmPd58aBrmmWCVItKOnxEgBgzvEOE0lTxKnhtJPyK6/zk+6mQprZjkh2qGscQ9DNaqM2kvyHlKnNu4vg5BSVFMgvTP3ip0ADJmjCa3E9Gg07DeCIno//RmkSfIQSgVXiBBaiKlAQSuu2ZbunrBxYyAbIRsA42OUZrokrZxpOPw8Ym7UPs16nVeyTUjEK+9XC+wmqi7QOKNj50dEhI/ha5JnZJZVyNRnjZhogcONmcM4AoM4SoW6nnFbfqZeoO0613JhjKs2wTkOFMZZPHBtSIFdYLzvumJmvkmpP0OcIxQdDuSh+AhyJ9UKQvE2HAQLjsOm3yy2aFU82ib356pjIAN09bYGsPct2GHGSqO3y3QeauSa2tezBz2P3sQH8D+SP/r4h7nGQv2GO7OJ/67NNTgFBhRQGGg2m8pioySyLIkWvGMp1S4UX6oQRIpK6txEg/y4NGCTjaAYleiYVm8gQT7kunyXYQz8rdVcl6RPFYgtyZPowF1uVDKxcbcdmAC3wFgaYSBM6AQ52Bq3O6pArkPbCHYi1gFnPLARhTU5tQFUgTeOwJWe5k26Xu+PuOgefi/Pz/RAmDVEg0VcT1pjbx2eLTTthRJCpAakXS/ZpA0+gP8lF+sJlPU7UCwW6lF2ukwhfM29fFRau0dCbUUQKrhxoU8I4ScwvqdjIdxrLiduUN1B87m63IktZitW+qa41U64QeHntiAzN6SabUt5pWb5kLr2XIEwFl2s/A1lgT6MdkxAgUE9M8Pu8kWZlI+nCGz8VSlndctRPO7nNN8Pzc4V9nGey9MQvQgp70pmiJdhsQBppy75lfJLqwZF/2NmhLSTg+5aOtgbXRaebpWaf82SthS78tL/Q3y2RVVKJIllz77vZF/O/u624KH7qb05YFdEw3vvXEdtvo9onSCBVGTq4/P1VnNpNvl2WrG17wrdb0Fsa3ExSq82MA5b5nuiNAoEgBYgmBALU8W/d/FM3EZiv0djKHIVVXKvGF5pr4oz9/pdoIIQM2mA5SJoQIpNClvb4Q8ymhAzATwyEvxtCDQxgO273uKli8joIk2mh3Q0+8rKpIk5oWlTlWHdohdV7L1Bq/Hu7CKzj021JxkR6M2yER1YvgoEa3B32zh31rXQzo/sMWAnDx3Cncx5nNHmzgsxGHJsNMYKmgs4mWOrwW8/4cdvAOGVdz0JUUlzCokGaLAcmt9rYcGGtR3Npk1IKK9G1tDSioyxHS79bXnK4g3b7QaB19STevSdRcl1hHVJToqlx/J34S462DFj1y/1Ls2P9wLX0l2MJkbTaqxKZoHLYK5pEEtR4+ioo4ytferNl2OIIfHlfEq8f5IAMXHCkbNHsgnrjjgehle8SX0RFRoZAQUypDtwScjalg0BHX8TE0a+Qfd4vlutp54/u+x34PgnvpiSEGkEw3fIGE5rzd4V5mc4OEzQlWD0Xb839317GWXl3nkgB7Aaqnr9LU4ZNBre1U5xz/8IMnXtMpQMNzn40zgHYdtNSc2ZwJLLbcAF8MIUny0fdgy3HT+eNUXJ/6fOGEzyNByN024s7fd3x+ZUIWgjNz4aXSSZi5phLlqmI4XUu6WtG314z4p63Du3zp23Ir2p0f+qbd322w2ZzcOzFLprfldKhLWr6A4udZUmkrKCXbgGaMnvgCjcdrn9KHRxxN8f+4itL+cVLtaf34F+ZsQhLSFff2CR/WX8OZjzijyKtk/xLcb5qsXhYcbOhKEfR84KOJLsbo+LJbv8ELr8OHLt9Yg6NdapPwJobkMZZ8B9fzikwulUgX+l122Av3XcI5UlebrJIY9DGaBgzCzi5YsGeoVJhnkXUYnRwAOFUbScUI0+5Uafqc+TYRCcwLnYSiI1lRGzGB3allek+IDam04XGiFZmwopOQAb0GZp5gpuiGV6CT4AF5gb0sNWSZrQY7X7woKew7OOAEhpLVIGCjDQJKkILA2C1vubmB478r50gYs/qSfpVOpKqwTAoOsoIAUWsQeKRJ7DvKIEwlhNsMvpLrD+2ATyotfrFdbc1HRlEgzRyOrYYNWT3wWCJpTPp10kdDdDFuqGX9jWPMTUX0Dhc5PFk1vTFpSO+YY3BLro6BY9q8X2AZ5lenTgDbeHSTsg32h5WMlft2T6bA+cFgH6st+jaTazA0cR9da2RT0cGgDv3y9kLpREDlVebuncDeketLwwIISUT7x54wmtf7qJJL6FHUroEuNtEFwh41+6q7OopH7npr4hux/Zi3vQG5l48Gp0Gfz0O4tt1OPvJfjuzMI98fP74puZc0of5+qsc9qp27ls31T73sfdS1L/lmrjvXN/PsG8izg9pL/U2XuH/cvQ+qD4Oepd/ej+xoAZ+NmQqZ21GAtsqlZ22sypw+CsdexZ4iBojNafBbYW9mHNormFOSQUkBwyzJZPkwTKoFx6zVfg2xlbwhVfIaFlZ8Nxi43PDi03vh0Ge5yshTuHTvsUaUYtWSKnqW5/0bi/zWCh6hRisGfS9jSwL6mhGP2Ucwy4cWe0BXhvq+SVWTv6gBgq1l3SXFjgvf7bVouums4liFTxiULs89PRFUBIlBNOTCx6MJqWZ7r3hrihKZhFq/ZekZOWBzEefx9XrCPiIkEMQJifwXzvyJneyVLzQnj65QTUSzs5fnH+luJk39KNvubOzWRacdG9qm1jv3dTh+ghs9J/ZAl3YYHFro1tpcNbinL72/Gu2Xgb2CvEfD3jSwocv8pQ7PYdzFhi3bnDqQrTlUGrXHIftn1Cngnvn2OuBSe5x/Z4xOLJ8cxJXewufwg8by6dL/A1BLAwQUAAAACAA8dmpdHB/BThIFAAAmDAAAEgAAAGJpcmVmbmV0X3J1bm5lci5weZ1WbY/aRhD+zq8YOV92G+MY7i5KSYlEE5KelCPR5aJWQidrwQtsY3Zd75qDVv3vnVljg4GmUvmA7d1525lnntkgCH5W93IxkQ7Y3e0DB2FBQC6L7qIQawkiy1f4XchUzZ0pInh4MjAT829SpzYESzKmdHnpBh2AH+DTZPIbADNab4tSO7WWr+Ht13cjUAuyYqV2IcjMSnj7+SsH6MKTVMuVsy9mqpALLV1EumhCG0CH8xVoKVOZcm++WmFR7lYhNJEXMjdo69LvYL+W7q6Fc0ovyQaYorHSzU3hCqEcbXQ+VWeCRWaEu+oD+yWEXzkoDXEU9fYHt+pPSQlzK4k7KA8+aVEQBB21JntgbP2my3W+I2mddzp349EEhvgaiaIQOzaNo+tXNyEav7556R/xy8eQ9vcB8M6Xh3cnGv3+jyTa719Xj5sTjU4nlQtIrFqujUrZllOJALPlykJDL4rhBTB6PCc1uc1ZF5/zTOVsG0L3Kg7hKuacDM0zYW2TqvtSa1lU1ryLRGnlkoRZmS1CWMlChkBlTHLhVsOJ0fiNWU00ZmcYXKpEEPpsDnsxnSaVGzWXXnEfNP3IeuRzPvTC7Q0PmyGQTnsDMXW6/gw2cfTqauArNyt1msn0X4CoqvJ+GE/G96OPsDapzLALcOlu9PBwO/kAHkhLtZEWrFk4WYTgilIWINOltOGRU1tD+rm3uT9+7bh2hRkVZeZeV72Ei6XF6IzOdvC0khoh9A2/Fwhd1u3WwXYpWE549jKVG9RdK2spxU0U2IdNZUiA8nJIMf1csWsveLUKxJVdPIo2f4gBvL+Oey1RuZ3L3MGtlx4XhSnOTR3cD7E9InqLfjdKswo3wT4diIigVYqAXz6E0GljR26VdZY1m7ztvm7KAztRQ+JSSyovzAZjm+Y+yTk1PQuIxMZbOS+dMvozSqhUFhQj8tj5Ovd8R5poPFpKl4iNUJmYZTLJ90KW8ceW32MYk9qtXmBG9Fx+kVhEow/HCqExMqQ3fm5Had9t1Cq1WR+H5yl0PY0fIxI419yzO2oGpDUIEK7kBDUaYWLwi6m1O5wKHiVtw7t9gZS2snAMqeVS6Tfo1/is1hwR8PbZFoVZV01ooxodte9mHuzxea6IRcpsLT5fyfm3xDrhZELT7aKaz0hFR5iQ/Qu1XjAvUxFQmf1pI/qMlD3UmfFqzgXzvAzah6gYqQ6XzWaICUmzR8t0+F6g1klBqRiVG+T2lH2/a2qi5SEyTJ5kZi4Im8PLgXiLR1lgp2lhNuUX8FWdgQw4w46yxCO5ERlra2CWWolEaPn0nZPDke36NVqJbMEuhHAEVJ8cj9QjP51mQPlbzH46FctZUr46IoYaDpt+s4QN9oRmK9HIrkQup4P+Af7UoSgf4YUG5xCr5EJgzYQKD8MKy6A0ToXcZFUdSPF28jC+T0b34xGPhHW7XLKjyY1zuX9zE8Utf4xt8TZDFwfax9sAjxAy2ubGStbHG0AIPT4lOn+8YPKYOg88g/yvjbswA3Yt3kCuZNUQ/+uYWwaw/Zt4ZIqu48fTgu+iNaKTw0/QjaO4R02DS2KLS2/o8tEb1MLPIDNLhQOw+4aIZiZmKlNud4YNCqq5zez4f5PROQ+5ppGIDxJ/J8OL0SmG/yd4vfVLcH1SOKcqv6om9YRYjPFzK7sj7DPHp93eY1SfmlcFxSc2M/5XB+AnFdjf8I4QursACcTrE17WvoPPj7eT8eied/4BUEsDBBQAAAAIAMw4al3zddUI3AcAAIkRAAAQAAAAbWF0dGluZ19sb2dpYy5weY1XbW/bRhL+7l8x5wIHMpFoyUkurRIHcI2kLa4vB1+vxUEQ1CW5khYmueyStKTg0N/eZ2aXIpUmhwSIIe7LvDwzz8zs5eXlvzqnaaeLWruGNtbRv9suN5beHmrrWoqqrqyPV01m8Dd7vCZbFcdXVFlqrct2MTX4tdNHUhDTVaadtrppVVpoUtVxv9NOJ5eXlxemFHEijVRDVX1xcZHrDTU75epKN020deoYLy4I/3DjF+WMqjJNdsMK6HtVFyrD0oJ2Zgu5dOPv4tfGqdKr4ctBFYyVT6fbzlW0KaxqIywmJ0miccIHk7tf1v94/i5OHpWL4jiYVpvsYQ1nXLsWDZH8bSZUO5vqm+ezCTWtrm+eD1bf7axttBgshykyVa4PMf2g2tvqaCtNIhBQO1su5KD3oml7TzfG4eM3UfIb9TpboxuaviGtXIGfbSIqbyktOueO4ZLXuTWPOKsoVXlYL1XzgHjkIl87hzDjV0WtU4+6aPDhbLfdyfZ+ZxG8rDD1pxBNoX4if9cNogAYpvNkJlucQIZMRU5VWx1hqzRVVOgqYBcH8GIPXQCO/zV9PCUXOCbZY3tnC+vC1aVZhVj99P1P9+v7b76+/ub+9r+IVi/CIJvoTW/YE4JRLxZEX1DZAYOUnWL0jqe8aS2V9hGZuldHCQgwCgCHpDrJ/ojfBj6MM4w3Quawomot2K8Z+4idnjDMN7PkxYQetK7XTrXG4nuOBcackyzD92x2PSQUX1z45KVZkswp+nbya+yNleTR21JXLexFbnSmar+k2dX1ixcScp8j/4SyRg6XCpFhotuKoqdkW6ZRbXSGdHlzM7KqT8VCuS28QrQ2pkCeNKUqCjEWGZk7C7lNrbOH5pQqYphUiz5rqtyUaquF9LmRQyXQY8dYKTCRtUIBoAobOJTgQxdR6UOLsGIdeTYkS0C8qpP32tlG4E0Q1VpPeFFw8JdRllQTpDZdGZUTr0loeYOzymfqnJU/pXlIJ4YCt5aG13xWT8AowKdRwjRA0pGIjtk89QF6T7zapFSHKF6dXIY20zAd2AA+7nWl27WspOfe/xHc/4BScznIlkpqpyp72IK84HYI5KYrkOK6ygpUolxY78N24wPHoji2Pg9OkMptnPHWMN7mtHdEtA+N90BKeiSnB+a1tst2mk8c4TVcjCVgBNMPHy4cAy68UPqgLWcrmgLncPzD3bnsjmle2fakU/WuS3xjen0zsAmRgBDzXi/OiFwu5cIK9v7sOj0mcVTiDvgTJ6ppj7WOhnTy1GZerU2zRg/pGoM+F/EKsoqpPLsGlQ1+fDWi8K0vwlNfnKUUZ6g6zlRbUkVpUZvgz44/AUBY0Thw9IuGa/lGGRR6vaDciveuqwKl21ZuVmTagYXwzDc8sQ5FcX79Mk5KVKUoZHjwt7AM2EZQQ4pv+q5cF6ZlH6uoWvctSDrX4Nc7WaalLCeJrvIVbS3n6165/FXoXLRE2ZIjssv5ytsUOXYRGRondC+2oEbtrScmFaZBj4ykwvHd0NmMZG8KtOLB130Ob/lC5BkiF0DnYHfwlxV/9Nx0zv/PUYHMidzog65RB9eqqHfqQzwmfNhv+Sv+9wDTvU47U/jWy9wk2Ufh/r3TPN9EbBIX3G8Pv4ZCrxwGk2ZU5RmYWjWNHgqt7VouUD9ipFghZXurTiUDKWnyw3nF+gQ+AzkgdIlbTIyTV8uH1ecL7cFcjNnKV/524+E6J+JI3wBdrzDEAmf6rmrRURrDOreolz3m2xHVeGkKUqImqL7TMddkGjpVysjDDO7sJqRzNKcMg0aHXoj4oGa2VlJ0qJI8KahHa3LKlXu4Sh0m0JZ2CiU29k2WLUq3i9CCEcrDs1fewr57c3i5g58iqOC1dzhJkglJJEdVR249u45PhIaKT21/QQ26Lca2XGe2ahUqLjcizMqh2Dd203pPU9RLDGFd46e94PjI4woDUIoZyRYmnwb3/anPb/ByGSZHPtffwPGvXsafsn4f+h6eD0iyEnNTgYEmEiETehmP8OHGp6fP5VpmnfY9CZmVPUTLj8jY+LsZU+QkT1I5G1rqs3glWUtXtBfJAhSsH5mPmYz+3vvz2vsTf0bguObMkpdQL0KfsLk8W0d7aMPsN+HxGao5hvy0IumvAcCAfm0OPKRjpODOzg8tH6NXfoLbG7w4ZGCRgCrm5VnEmM3QHM3RRh9iQCjIPaGHUSHhAwpb6fYjnvSX1VmR7D1hAU9J5lr48v965wY9ZaBhlG4ntEfDHD2fmKtTHDvnq88vcHBPB9pRlGHihQcZBtD4U48UHkZ4kmCc4VQ/SUDdeGG2Ci7hNL8tnOZpQSzDyw2dIhg6TCLIozjm+fEvm7Ow6Xe1q20hJLxhwd/9+PPb+/Xt/dtb/0x5jXEHQdU0bN795+vv7rw5RwB5mHEKuvGEhOf21RVhxjgty2i098vj2LjlcbY4zhCXHUtaHPjnfhXi0GrGShXrTYEnLmgiab1GSxpC8QNmBVIpErFrw2N22towxHgaZDumj6+vfy0yaGx275/prVY5nlTo9O/AvBRtb1eijsJYDtnQ07hkLIduw8WdWfoeSXaycLmYchSH7/liNWo3olmoe0ZbT9kv45j+h710vJee9sY9i+UkqjpG8XnHwmJdo734FsIZDowgf0ppvORLq37KOh8o+tOymcfybMh9BsCSiz8BUEsDBBQAAAAIAGs5al0rbyBziAoAAHkaAAASAAAAc2NlbmVfaW50ZWdyYXRlLnB5nVhrTxtJFv3uX3GVUTbdg93YJiCWHUYCJmRGIsmKsLNaIWSVu6vtXvqlqjbG82F++557q182ZpRsFAFdVbfu69xXvXnz5pN61FQtNZVGW51X2lBaFI+UJm59Q8po+u2znLGhzvWQ8qKiUtlKR1TklFTk5aus3NABfSl1fvU7VtMNH6OsiHRq/WAwuGPy5JnClbak6CnRa23eWdIbTSuLtaogW+JihSPVqFhVQ1J5ROulqiixFBW5JjXHOmkVLs8GRJOAwiItVgZyxCbJFxq/ikwkNVqlZAp8yb/Rz6RtlWSq0rO4MHphilUeef4ZzdOVGcUrm0CTbqe+uCHC5pBswTcP6Hv/6QiClckzDEGhMmbjzK2NLXIYwHFyRuWNIoXSKk3BaBrUx+CNiuZwBUVJHGsDP2FpsaxaJlBwqUxW5InVrFapTJXABDcXlzWHw7DIKwOvEX7mFrfA5Gtlos6z5IWqLHXkg/dRwP5zTFqrulOQhz9YL1vzlnOztVElM2+PQr1inYuN2UlOEItvzdB5YhEEFumyWOmq0sPvt25rkHTDgjE2cGEEXRStkzwq1o28qY4rWNZktvsU3zBaWeX3orJdqh4Re2IPV6gcwpQzd5h1tgWuq2ljEAGOpNZqs2M8WGRu2FjQncUE2+PWy3mR2ESMohz3MgkrwJvmeglVhC3QGC5nfFL87Dyy0JVTyqoMP3RuC0MLo5KclNsIsWEUEF5UasFcTwJi1YoY2wivOE3CR0ixz8TgWumsLIxKA0S5p9JyqXxEu1ws8TyHMyVRhEVWrioJSKhtM/BbUpJjzQ4GH+DxDdjBNGBpK7htUS1HAswiTUHljekcMsV+HW0uF4Uwx1zD1UoOlWw71jsg+hVpwXIwZEl15lTB/+noF0LcQtHSFKG2FiwD+g0O0mnK6SdGOEaNUTjNKOdAzhj/4IwWFVCLQ9JoFwOgUWGdYBTleg1xjA45NQSDN2/eDBK2UEUuE8LqeTkYDCId06yIIi/3zwTbOfRL8srL1LN3hKD3fVk2Gn4GACiJ8eMtTQnZQuPPA5rU1+zLX2Yxn61OkSjZI0Mys3myOP9ccI42M5vBXPJV84aUl/uzXZfmyLuGFpr+Rv9MqkTTdDydDOndRQlDPgt/uuYcct0RX7lU+aG9450/pGpdcIlAYg+E960oyDgrkM9vP17SOAgmyO6I3a0MySDakx8RywBSkx/nKnysuRudAXtRwC5gRrUbwqepfC6HtIbJxUABwrPU9akFVp35AuizKbWXl4FIdzT16ZCmx8fBWM4qnMRemCalVxt6PKSJv4fOnT9ihvdBEAyJrf/gPDzHqkBBvEQI0CzJPZbPpx9hjcnY3w67HyjTynJ1O8P2+MRlrOPTty5oa7NpYwpUD6bHAfUE2P55dPyWPLvJcahKQodsPlxjzXaSCEZEFodHSHl4SKdApRxl3MVz73pIl9irYcT/5mwU2Djg1O7BIp7hA/6u1nL2un/2Gsqqo5YAlvZwGWCuR8d+R3PZp7kEjTehEQj9LUpZnEsu2qa/5jgD4wP2BojZ3yP3N1ZH7W38fdmRNWFYe/s6r109xClnkeuhkw1WwZ1Dkh9m7jebM7cnx7Bht8L7ug5lVy05/XvzBRDYxedoEgQHE5TvwiL8kFPPxdM3H67vGq+7jJ/06ojnMhRSL5zflhtGix/Qn5xR9ZPOHdfX4mRR2zt8qjiejRNsKGtXX26+3M4Qs9OPtxf/eRX3CFRG1iIAbHNv7BY5+FKde7zpVm6GdMvMsHB/hn3g7ejB0cDOsjzy3LJ/1mz0rShMvcZH3g08eStouIG7bx0UxLPvAzhvNGEH+rXlux4pXrRps1a1qUfn4+AIhIgIlPckrs4nuKhz0ScudVsZCkFnUGsoXML2CvVR6glatirVjdea5st/0XS9swHFi7OdzPian7ImmdHPOPd3dyRuZaefzmks8RzYVeb59BNNxuOzXYDHC1mJZ6ma7zoeloHtkP5gyXFw3Hf3CpXr1N8FBVrMVzEx38dhL7ReuSXAVMJZ22M/HrlL42xIMUNN5L/PHhrEDbsVW0VYqMHgRAHVnKlEpo7Efb48/wMKW+l6de4K4GcW9l+3dHXx9Q4ZBR1XlNhK5U1XkOsVXJz6rpXn/W1gIKpXri+0j2jMahwo9Gl6M0rssuYqQ0tDuVZ2yc0UWg+00ph9mJCrNfczFw1LR1NxLeWi3tSBfWVUpcjNriQLuF1t+0Obwko/suM/OXM/OXvo1UBvnsnKiCbTU0QG0NLCjwvZMYKuDZ5eHPVvHPOFO4GMa8d8a8y/X9w64WtPOKLxw/ebEqmfkSq5Rc4KyyMAHLLpd7zvrHBl09V4gY9FBpf2uEU+xyQ5xvKuPJblOQTUnFyTPbrCZCPINpUCPK2buZBZiTIHrbqNFK5Aivpe/3NEIV8u8hxQ2BO6kUcmYqiPyHwZk/38uBVrQtVFGqJsimjbF2ltz9OvUTLRbafKeDtTHneZ8WtotM5fTH9ePf757fyHXFgA+DwJoHojW/53ZRnaMrdtT4ScGx3bb82Qu9nwteS33G4LGx9xszSbN+Xwo1pheFA5982eU96DDzhvcMc0ZTyiN5vCdcva+7lW5pULXNch09MeB2xfPeE4wtWT/t1irA4SDasfgc2TulXh9krqQ9tFjtHG6bJvXpcUxI9FvuD5SVVtR+mswPjsIZ0Z95s7jgNnqD7yOIy4t5Jf8cKXtk0+bFOA+1NzjSkWa6f69jDVG6szVbqpoeiBbIisKjlye9h2zZHcHdCHtv3hWRajHne9CAuNmizPEnmlwobN/42rJo/O+OmsUW43ob4yj0TPWBT/cFYZH+PX2m1ssNHCQJY+OQTU13r39xNxffT8MKR7xgD+3jw8+K0jGYeoNeVFHCd5Ldle/H0CBNdDoA0RV5hIm09FpM+Z/PLL7S8fbmdXXz5/vbv4fNfs/67SlT4fb7PagrztUM1xcipgHmxbTbBse6lV9EF34++Ovu7ZY2fqDVV5ftLv0K7cWC+HUQufdEre6WiOqZ5LrVuxfoeVtkZytopTVXE1rufRgK7kSewMZfdZ3mKMVpGVAZ/v/+Z+uhH6OxrqZcntNJeGXasuOqtOgvp0VnMEBDFXeL0m8XRfD4c/+enE85Bmjt232/KZqEF810ZO97SR43pCfgLrZYm262WbzjMuHMRyvj+dnsDJYJTpCJqwNGpuvSdo2C0++fyvdjwE0gsDz+/6vG4h23Z+9lh37Vyw3Af0cgHd7onD5Ot096ExXswakJ3fmZV2Wal5SskXO88o7jVDwIOQSlyPZvjtzRMjktRYfl84FIEPkaQzrnFz3T3T/UrP9O8WP9/7ysDFiJvZxTe8YMSMxr98P/LZ2T0juOenb3sfkdt3Bqt9HvK707vNRdNYOO/5bb7lhIhJl43fgU9Wz18O0Y1ZrDNL3FWfraLTFpwaHP2y1nKuocKh0Ge8ZcedZMTVriHrm6fVZbF9m4RKLs9QuTwHwb4GZbbIAkBfrdJqhg3P3yLo1LNoE7Ef5IVBNfPG3ImAh1jS1i3N2fRhH352H2nq5pj7iP6zFRgIn9aM6oWl6lD3dntUaUP+YoQc/A9QSwMEFAAAAAgA0ThqXfogYc0VBgAAEg4AAAsAAAB0ZW1wb3JhbC5weZVWXW/bOBB8968g8nJSK6uxix4OzqVAmqRtgHwUrnvBITBcWqYttpKoknTs3K+/WZKS7DQp7gIYkcjV7HJmuOTBwcFElLXSvGC8qHPOlmsjVcWWSrNa6P5S81KwklsrqxUr1UIUhkXVuqwf2Et2U4vq9C+mquLhiFlhLJ8Xgm2kzdXaMqt0lsdpr3ebP4wYZ8ZqBRBZ8pXo70Gy6J0ci+W1sAm7upjETBqm1xWAmbgX+oH5Mni1YNIaJhYrQSFim4miEBW+miOf4FkeIqXpLUQmFwIfVAuBMhcIKx4SZhTLudQOa4kChDbM5LIshU7ZJAcqKlpjFVqU6l4YZnPRBLDbi8nHmy8TNi/WWlP1WoC4UllQNur1GBukTNVWZhhdFmrDosDQ2cXnmG24rj1ercW9VGtkLpXCwCKQryqrXEAGfBTsV3ME4GHqxgmijUatVq+NxQApwDa50MKFudyYNqJY9jNVGYkowEWQFRCLV3OefacHluUi+x47Nvzn0kInWRTIiT++0gIckKIOuBIbX9NvhqlNRQtZyIyWzyLO5nLFFtK4j0rKd+wJynIOohMm0lUKH+SUDeQSgbKKaXmv06B3Jlio0dXkBtpiaYTDD5zAjGNKOyNUCpGib5zgBV+lvYODg54kY1vmzcoRVfd6sMWSzVaaP0R6NY9HbpkhMLsfulct7FpX9Jpm9/ZUFUpTcOJGTm8ub8az8Yd3ww/jk7/jBpEYj7gDhhnDf3ARMqCcM1GZoIzNucWeghlquaUNpZagpf+WzVn0MblNhjHFcft6mNI6dmoCYgpzZT+lusb622LIJZEsV4lLN4OZZlbNyHRdOZ95WcPlXxH2lUWtIf32MTXPBFxh2Ra73K1tGxPjcEchKzhiXXem2DNrW/EjUvOEbeAHpEtNzmtxNxpO3cRqm7DVA6aqOi2FyVdaLiI8c02miTag0T7U4hhDgZQ4Yd18/sR8/FhG7GXuGEm5oeBoD2zVLHKXqbs0TRN2OHXFPTc9mCZ+nzz+o6QX15Pz8ezy4vr8ZAyhlF4IfYVud0yT727GZ5gdn3+6vDg9mZw32vk8qlrCyvB+tJyR+yHkbLlZwPRyVfLjQfqmE9JHjBoV+m87LYEVvhy1Laf/tpUL0yk78R70nlwpanhqv0d1vTdTEMjlpZToimpd0H7bbx5coyMdOQzY1XLawlG9dU2dBoXWCr127OQx7DBNB61pCHfGqXF4E7viOyn8WiHZT7ODbtYBIYm3lPmhbRRobICgZ0h0dziN2YsXbIihvaDBbtAgBO35CthiW0f9iFK98sqEsCdc1mxN3/DRjetIHKIjDhK2DVpaX3BWyDqKtqzPxGEM4EgM/DMqR117NVj2wv2i1wgZ0kvro7URsxoHndvaIW3iRA0dg8zszhH/2HYsWJUbcXyY/v4m8f1/Vii8Dg+b11zi9c2OBW8qtAwsaRTqMu1Z8eh8o1vF8x3D80/au47TVbVTNbXUEAzt29hH63oUS/sJob/cWz4ynK+N/fa427VXc8p5yfjcROHLfsdrQHTtAnHEKhRytUCxQXqI4F0/NFx3NCdtnv2WFgXQF029Lxs8PxFTnraMX7ix5sZELgqyE2kmWMA1qwRu/r4JMtOF7pjd0TkzBTxuXeFDX1lYJE17eaC0xOnukUZtl5RLH4peQLGjvfaJHHdyChiPjOenam+icXA++v4bPiX0p0G7LUFD36Zhxe6pTdgM0pNjwlHQAoZlyl01gLa76Yz4sXb2anh1Jjczd4OYy4XUwl2YeHE80WtB7laQ2JhjImSPc2wMDzLCqWssXRMcC+6WELvOCXRV+n7d3dbpSn20k7f7fC0r+wfD7aXrv+0WpZjnTu8Ki34sOaSs2J9s2GkQCLnjT8jmHMHJER5j2uwjINOBiBvyjb85v3fbFBdHK6Jm5tMEp+Tl7P3lze3s0/j88/lkdnV+dvHlypfiNCN7+pvd0mdbUraOBZ8xdI1nne9vFVW8q7wzrRdptGMFPxKhF3aEKLuv8U/kIL8/6f5bHdjTOCL67vc/akI32OsYdyiSug6nUy1oATcSQf/guHF3C1QUT3v/AlBLAwQUAAAACAA8dmpdiljwecsCAADyBQAADwAAAGZldGNoX21vZGVscy5zaK1U207bQBB9z1dMDUKJiq8QJwGpUiBueYCAQpF6UVUt67G9wt51dzfQqPTfOw5OCqSVeOBhZXsue86MZ87WG/9aSN8UnS24Dbzh3gHYAuEORV5YA0ymwFWKwDTC+HSWjCef4ehqOjlNJiAkxQoDNeM3LEfotln+Mu0WZaq03/NgNm8DlSwXYBVodFN1J0vFUlCaIrXIFh4xAPD8DC0vvlcEWhrPFGRz38HSiA2fBd0kc6iEMc2zeyRmmE3RQsWsbSxvoVbaaibsLqydOUrUrITz6fTTLpwxO5YLJREimCXvk1kyPU5WNfceii6Q3xi4PBlH/fjy6uyS2CVEtIXnqiI2mVYVfBD2ZH7twYWQEtOmvKaBFFAJauCy7jtGRwtrUQLLmZDGQneqbiEKorjndQwxdLHDU3C2u6nQklVIr4HTcw6huiELuHXb0BXNzldw101e1/kN7u/hF+TCAi+bAgtra3Pg+2QpiCXR8r8UKPMLOoO/ec/v2dlZ3uEeb3iWjVFzC9kgGIzivSiNhyHyLIzCOMw4suEo6MfDaLS3j/0R9sND+P2E67r50YvY1j/cBZP547TNizboPvKtCQfENxwNGE/jNIzZYD/MBmG/PxwGIx7EvD+kMxhh0BDO0XZ7RI2IG3BWc70dOQ+c+VyX4J6C62ZMlA2uC656Guhsh0571Qv/gq+xRGbQ+Kvt8G/DtddtB9zFWtGChEHg1baA5+7G+HqYq1Vagfafga78r4vabmsLGu3ve0rKn3AtNGYS7fLrf2ib8/JPMC/wAp9axh5illU9/ewgL+iXTs8/JgdPFKN76Z6yaygFR8mxB7TgGe1lRRhz3YhDI3MeTBRIZWFuEBohoAmsaQhJMxkJpkih1iqdcwt3xL0ZT7aWiEY7UHNBetWCeE6nCyQPK1mmeTcFI2Uy8wpc/kimoNc4H6izslxnLDVWYOp0/gBQSwMEFAAAAAgAM3ZqXWnK1VDjAAAAUAEAABIAAAB3ZWlnaHRzL1NIQTI1NlNVTVNNjztOBDEQBfM9xVyAVdv9dcoBCLiB3e6GCZhdrRzA7RkgIX5PpaputWGKtOqjGxRgtZo1zX24RVcSD2NER2taFE2ag2IM40jxbXveXyNfYj199LX24+16X+8XzkZ1lFlLk5zkOHNaNESeFN1CpzIVJCEGB3LwJAKOKdLPq/7D3m+P9ej7+uNaSi0JkHwSBJpK4abQzbq2wkYCMD3ViUexDl2qKCP3EmPiqTv2R+QR63o7js8LR7NagsC0YqGqqDII1Y3HkACtg0qw/YwJZwpXSuXh1JNboW07q/vxdTui/gp+A1BLAQIUAxQAAAAIAIc5al3mW0/HrhIAAAcyAAAQAAAAAAAAAAAAAACkgQAAAABzdHVkaW9fZXhwb3J0LnB5UEsBAhQDFAAAAAgAPHZqXRwfwU4SBQAAJgwAABIAAAAAAAAAAAAAAKSB3BIAAGJpcmVmbmV0X3J1bm5lci5weVBLAQIUAxQAAAAIAMw4al3zddUI3AcAAIkRAAAQAAAAAAAAAAAAAACkgR4YAABtYXR0aW5nX2xvZ2ljLnB5UEsBAhQDFAAAAAgAazlqXStvIHOICgAAeRoAABIAAAAAAAAAAAAAAKSBKCAAAHNjZW5lX2ludGVncmF0ZS5weVBLAQIUAxQAAAAIANE4al36IGHNFQYAABIOAAALAAAAAAAAAAAAAACkgeAqAAB0ZW1wb3JhbC5weVBLAQIUAxQAAAAIADx2al2KWPB5ywIAAPIFAAAPAAAAAAAAAAAAAADtgR4xAABmZXRjaF9tb2RlbHMuc2hQSwECFAMUAAAACAAzdmpdacrVUOMAAABQAQAAEgAAAAAAAAAAAAAApIEWNAAAd2VpZ2h0cy9TSEEyNTZTVU1TUEsFBgAAAAAHAAcAsgEAACk1AAAAAA=='''
os.makedirs('/content/se', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(B64))).extractall('/content/se')
print(os.listdir('/content/se'))

In [ ]:
%cd /content/se
!sh fetch_models.sh

## Upload a clip (5 seconds is enough for the first test)

In [ ]:
from google.colab import files
up = files.upload()
clip = list(up.keys())[0]
print(clip)

In [ ]:
!python studio_export.py -i "{clip}" -o /content/out_clean.mp4 --max-size 480 --engine clean

## Optional: background scene and the reference engine
Upload a scene picture, then run the second cell. Compare the `flicker` numbers on the finished lines (lower = steadier).

In [ ]:
sc = files.upload(); scene = list(sc.keys())[0]
!python studio_export.py -i "{clip}" -o /content/out_scene.mp4 --background "{scene}" --max-size 480

In [ ]:
!python studio_export.py -i "{clip}" -o /content/out_ref.mp4 --max-size 480 --engine matanyone2

## Download the results
They are in the Files panel on the left (/content), or run the next cell.

In [ ]:
files.download('/content/out_clean.mp4')